# Pré-processamento de dados

**Objetivo:** partir de dados sujos (escalas diferentes, faltantes, categorias em texto), montar um `ColumnTransformer` completo e medir o impacto da padronização num modelo baseado em distância.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Um conjunto de dados sujo

In [ ]:
df = pd.DataFrame({
    "idade":      [34, 51, np.nan, 62, 45, 29],
    "colesterol": [190, 240, 210, np.nan, 260, 175],
    "sexo":       ["F", "M", "M", "F", np.nan, "F"],
    "risco":      [0, 1, 0, 1, 1, 0],
})
df

## Pipeline: imputação + escala (num) e imputação + one-hot (cat)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num = ["idade", "colesterol"]
cat = ["sexo"]
num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder())])
pre = ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat)])
Xt = pre.fit_transform(df[num + cat])
print("matriz processada:\n", np.round(Xt, 2))

## Padronização importa para o k-NN?

Comparamos a acurácia (validação cruzada) com e sem escala, no breast cancer.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer

Xbc, ybc = load_breast_cancer(return_X_y=True)
sem = KNeighborsClassifier()
com = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())])
print("sem escala:", round(cross_val_score(sem, Xbc, ybc, cv=5).mean(), 3))
print("com escala:", round(cross_val_score(com, Xbc, ybc, cv=5).mean(), 3))

## Exercícios

**1.** De quanto foi o ganho da padronização? Por que ele aparece num modelo de distância?

**2.** Troque `StandardScaler` por `MinMaxScaler`. Muda muito?

In [ ]:
# @title Solução
from sklearn.preprocessing import MinMaxScaler
mm = Pipeline([("sc", MinMaxScaler()), ("knn", KNeighborsClassifier())])
print("min-max:", round(cross_val_score(mm, Xbc, ybc, cv=5).mean(), 3))
# O ganho aparece porque o k-NN soma distancias entre caracteristicas; sem
# escala, as de maior amplitude dominam. Standard e MinMax dao resultados parecidos aqui.